# HW2: strict greedy inference

Use the selected 42.48M Transformer checkpoint at 50,000 steps. Generate one program per question with num_beams=1 and do_sample=False. Syntax retries and execution voting are disabled. Strict greedy on CPU scored 54.06% (519/960) at 09:26 (+03:00) on 16 October, matching the earlier retry-enabled score. Generation took 99.67 seconds on CPU; this is not T4 timing.

Run CPU preparation first, then generation separately. Preserve the official test.parquet order and save exact outputs to Drive before exporting. T4 timing remains deferred. The historical 57.08% voting notebook is archived separately.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU for file preparation'
from google.colab import drive
drive.mount('/content/drive', timeout_ms=180000)

In [ ]:
import os, sys, json, hashlib, shutil, tarfile, subprocess
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Keep GPU off during file checks'
os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
assert Path('/content/drive').is_mount()
prepared = root/'prepared/inference-42m-50000-greedy-v1'
prepared.mkdir(parents=True, exist_ok=True)
work = Path('/content/hw2-inference-prepared')
work.mkdir(exist_ok=True)
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''): h.update(block)
    return h.hexdigest()
source = root/'prepared/hw2-strict-greedy-v1'
source_ready = json.loads((source/'ready.json').read_text())
assert source_ready['source_commit'] == '8dcb74d2164a30c7b3169dcf7adf76114139761f'
assert digest(source/'source.tar') == source_ready['source_sha256']
with tarfile.open(source/'source.tar') as z: z.extractall(work, filter='data')
os.chdir(work)
sys.path.insert(0, str(work))
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.checkpoint_io import drive_preflight
from homeworks.hw2.data import fetch_parquet
drive_preflight(root)
parent = root/'runs/full-42m-100k-v1/step-050000'
manifest = json.loads((parent/'checkpoint.complete.json').read_text())
assert manifest['status'] == 'checkpoint_verified'
assert manifest['files']['model.safetensors'] == '01ba1ecb9c6eb4328142d9b164b58df1e12a7cb66c68e66b8015bafd8640c8c5'
model_dir = work/'model'; model_dir.mkdir(exist_ok=True)
for name, expected in manifest['files'].items():
    assert Path(name).name == name
    if name != 'training_state.pt':
        assert digest(parent/name) == expected, name
        shutil.copyfile(parent/name, model_dir/name)
        assert digest(model_dir/name) == expected
shutil.copyfile(parent/'checkpoint.complete.json', model_dir/'checkpoint.complete.json')
test = fetch_parquet('data/test/test.parquet', work/'cache')
import pyarrow.parquet as pq
questions = pq.read_table(test,columns=['question'])['question'].to_pylist()
question_hash = hashlib.sha256(json.dumps(questions,ensure_ascii=False).encode()).hexdigest()
assert len(questions) == 1919
assert question_hash == '214746f8d4bc918b6fa1fde8ca3185a518a60637f239a2c45efa9a8143f612d3'
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True)
assert sum(p.numel() for p in model.parameters()) == 42481152
del model
bundle = Path('/content/inference-inputs.tar')
with tarfile.open(bundle,'w') as z:
    for folder in ['homeworks','model','cache']:
        z.add(work/folder, arcname=folder)
    z.add(work/'source_commit.txt',arcname='source_commit.txt')
target = prepared/'inputs.tar'
shutil.copyfile(bundle,target)
assert digest(target) == digest(bundle)
ready = {'source_commit':source_ready['source_commit'],'bundle_sha256':digest(target),
         'model_sha256':manifest['files']['model.safetensors'],'questions_sha256':question_hash,
         'rows':1919,'cpu_load_verified':True}
(prepared/'ready.json').write_text(json.dumps(ready,indent=2))
assert json.loads((prepared/'ready.json').read_text()) == ready
print('CPU_INFERENCE_READY',ready)

## Generation
After CPU_INFERENCE_READY, use CPU or an available GPU. This cell verifies and restores inputs, generates strictly greedy programs, saves results to Drive, and releases its runtime. Do not run it in the active training notebook.


In [ ]:
from google.colab import runtime
try:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=180000)
    from pathlib import Path
    import os, sys, json, hashlib, shutil, tarfile, subprocess, datetime
    from zipfile import ZipFile, ZIP_DEFLATED
    os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
    root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
    prepared=root/'prepared/inference-42m-50000-greedy-v1'
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_load_verified'] and ready['rows']==1919
    assert ready['source_commit']=='8dcb74d2164a30c7b3169dcf7adf76114139761f'
    work=Path('/content/hw2-inference-greedy');work.mkdir(exist_ok=True)
    bundle=Path('/content/inference-inputs.tar')
    shutil.copyfile(prepared/'inputs.tar',bundle)
    assert hashlib.sha256(bundle.read_bytes()).hexdigest()==ready['bundle_sha256']
    with tarfile.open(bundle) as z:z.extractall(work,filter='data')
    os.chdir(work);sys.path.insert(0,str(work))
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
    from homeworks.hw2.checkpoint_io import sha256,drive_preflight
    from homeworks.hw2.greedy_export import run
    from homeworks.hw2.verify_export import verify
    drive_preflight(root)
    model_dir=work/'model'
    assert sha256(model_dir/'model.safetensors')==ready['model_sha256']
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    local=work/stamp/'step-050000'
    run(model_dir,work/'cache',local,batch_size=64)
    evidence=verify(local,model_dir,ready['source_commit'])
    assert evidence['questions_sha256']==ready['questions_sha256']
    provenance=json.loads((local/'predictions.provenance.json').read_text())
    assert provenance['num_beams']==1 and provenance['syntax_retry_beams']==0
    assert provenance['selection']=='greedy'
    gpu=provenance['gpu']
    evidence['t4_timing_verified']=bool(gpu and 'T4' in gpu)
    evidence['t4_within_180_seconds']=(provenance['generation_seconds']<=180
                                      if evidence['t4_timing_verified'] else None)
    (local/'timing.json').write_text(json.dumps(evidence,indent=2))
    output=root/'inference'/('step-050000-greedy-'+stamp)
    output.mkdir(parents=True,exist_ok=False)
    for path in local.iterdir():
        shutil.copyfile(path,output/path.name)
        assert sha256(path)==sha256(output/path.name)
    # Build the exported archive only from verified Drive copies.
    with ZipFile(output/'inference-results.zip','w',ZIP_DEFLATED) as z:
        for path in sorted(output.iterdir()):
            if path.suffix in {'.json','.jsonl','.csv'}:z.write(path,arcname=path.name)
    (prepared/'last-result.json').write_text(json.dumps({'path':str(output)}))
    print('SAVED_TO_DRIVE',str(output),evidence)
finally:
    try:runtime.unassign()
    except Exception as error:
        print('Release failed: disconnect and delete runtime manually.',type(error).__name__)


## Export from Drive
Use the authenticated Drive CLI or the CPU cell below. Verify hashes and exact token provenance before submitting once to the bot. Preserve the receipt beside the exported checkpoint.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU before exporting'
from google.colab import drive, files
drive.mount('/content/drive', timeout_ms=180000)
import json
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
output = Path(json.loads((root/'prepared/inference-42m-50000-greedy-v1/last-result.json').read_text())['path'])
assert output.is_relative_to(root/'inference')
print(json.loads((output/'results.complete.json').read_text()))
files.download(str(output/'inference-results.zip'))